# Detector comparison across model generations

This notebook compares:

- **NERO zero-shot**: nonzero-median NERO entropy-rate statistic, no training.
- **NERO trained**: Random Forest trained on the 46 NERO-derived features.
- **HowkGPT**: perplexity score.
- **ZeroGPT**: percentage score.
- **Binoculars**: score available in the newer API tables.

The analysis deliberately separates:

1. older vs newer AI corpora;
2. GPT-only vs non-GPT newer models;
3. older human references vs newer public-domain human references vs all humans;
4. within-domain train/test performance;
5. transfer performance, including **pre-GPT-5 → GPT-5**.

Important: detector AUCs are calculated only on rows where that detector's score exists. The tables therefore always include the number of human and AI samples used. Binoculars is unavailable in the legacy tables, so Binoculars results for `all_human` effectively use only the human rows on which Binoculars exists.

The newer LOC/U.S.-public-domain tables also contain many zero values for HowkGPT/ZeroGPT. Inspect the coverage table before interpreting those detector AUCs.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_CWD = Path.cwd().resolve()
if (_CWD / 'scripts').exists():
    REPO = _CWD
elif (_CWD.parent / 'scripts').exists():
    REPO = _CWD.parent
else:
    raise RuntimeError("Run from the nero repository root or notebooks/ directory")

sys.path.insert(0, str(REPO / 'scripts'))
import compare_ai_detectors as cad

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)

# Runtime controls.
# For quick exploration use e.g. REPEATS=1, TREES=150.
# For a more stable final analysis increase these values.
REPEATS = 3
TREES = 300
TEST_SIZE = 0.5

OUTPUT_DIR = REPO / 'results' / 'detector_comparison_008'


## Run the comparison grid

This runs both:

- **within-scenario** experiments: train/test split within each selected human + AI grouping;
- **transfer** experiments: train NERO on one AI generation/family and evaluate on another, while splitting the human reference documents so no human text appears in both train and test.


In [ ]:
%%time
results = cad.run_all(
    REPO,
    OUTPUT_DIR,
    repeats=REPEATS,
    test_size=TEST_SIZE,
    trees=TREES,
)

coverage = results['coverage']
within = results['within']
within_repeats = results['within_repeats']
transfer = results['transfer']
transfer_repeats = results['transfer_repeats']

print("within rows:", len(within))
print("transfer rows:", len(transfer))


## 1. Data availability and detector coverage

Check this first. It shows which scores are actually available by dataset and how often a score is exactly zero.


In [ ]:
coverage_cols = [
    'dataset','display','label','era','family','n','rates_available',
    'HowkGPT_perplexity_available','HowkGPT_perplexity_zero_fraction',
    'ZeroGPT_percentage_available','ZeroGPT_percentage_zero_fraction',
    'Binoculars_available','Binoculars_zero_fraction',
]
coverage[coverage_cols].sort_values(['label','era','dataset'])


## 2. Older vs newer AI: main grouped comparisons

The rows below directly address whether detector performance differs for older vs newer generators and how the choice of human reference changes the answer.


In [ ]:
MAIN_AI_GROUPS = [
    'old_ai',
    'new_ai',
    'all_ai',
    'all_gpt',
    'new_gpt',
    'new_non_gpt',
    'gpt5',
]
MAIN_DETECTORS = [
    'NERO_zero',
    'NERO_RF',
    'HowkGPT_perplexity',
    'ZeroGPT_percentage',
    'Binoculars',
]

main = within[
    within.ai_group.isin(MAIN_AI_GROUPS)
    & within.detector.isin(MAIN_DETECTORS)
].copy()

main_table = main.pivot_table(
    index=['human_group','ai_group'],
    columns='detector',
    values='auc',
)
main_table.round(3)


In [ ]:
# Sample counts actually used by each detector.
main_n = main.pivot_table(
    index=['human_group','ai_group'],
    columns='detector',
    values='n_total',
    aggfunc='first',
)
main_n


### Heatmaps by human reference

These make it easy to see whether a detector's apparent performance depends strongly on using legacy humans, newer public-domain humans, or all humans.


In [ ]:
import seaborn as sns

for hgroup in ['old_human','new_human','all_human']:
    z = main[
        (main.human_group == hgroup)
        & main.ai_group.isin(MAIN_AI_GROUPS)
    ].pivot(index='ai_group', columns='detector', values='auc')

    z = z.reindex(MAIN_AI_GROUPS)
    z = z.reindex(columns=MAIN_DETECTORS)

    plt.figure(figsize=(9,5))
    sns.heatmap(z, annot=True, fmt='.2f', vmin=0.5, vmax=1.0)
    plt.title(f'Detector AUC by AI group — {hgroup}')
    plt.ylabel('AI group')
    plt.xlabel('Detector')
    plt.tight_layout()
    plt.show()


## 3. Individual model comparison

This separates the model families so that pooled averages do not hide a collapse on one generator.


In [ ]:
MODEL_ORDER = [
    'gpt35_old',
    'gpt40_old',
    'gpt4o_old',
    'gpt40_new',
    'gpt4o_new',
    'gpt5',
    'claude4',
    'gemini25',
]

model_results = within[
    within.ai_group.isin(MODEL_ORDER)
    & within.detector.isin(MAIN_DETECTORS)
].copy()

for hgroup in ['old_human','all_human']:
    z = model_results[model_results.human_group == hgroup].pivot(
        index='ai_group',
        columns='detector',
        values='auc'
    ).reindex(MODEL_ORDER)

    display(z.round(3))

    plt.figure(figsize=(11,5))
    for detector in MAIN_DETECTORS:
        if detector in z.columns:
            plt.plot(
                np.arange(len(z)),
                z[detector].values,
                marker='o',
                label=detector
            )
    plt.xticks(np.arange(len(z)), z.index, rotation=45, ha='right')
    plt.ylim(0.45,1.02)
    plt.ylabel('AUC')
    plt.title(f'Individual model detector performance — {hgroup}')
    plt.legend()
    plt.grid(axis='y', alpha=.25)
    plt.tight_layout()
    plt.show()


## 4. Focused GPT-family analysis

This removes Claude/Gemini and asks whether detector behavior changes specifically along the GPT lineage.


In [ ]:
GPT_GROUPS = [
    'old_ai',
    'new_gpt',
    'all_gpt',
    'pre5_gpt',
    'new_pre5_gpt',
    'gpt5',
]

gpt_table = within[
    within.ai_group.isin(GPT_GROUPS)
    & within.detector.isin(MAIN_DETECTORS)
].pivot_table(
    index=['human_group','ai_group'],
    columns='detector',
    values='auc'
)

gpt_table.round(3)


## 5. Transfer experiments

These are the strongest tests of whether trained NERO generalizes across model generations.

Examples:

- `old_to_new_ai`: train on legacy AI, test on newer AI.
- `old_to_new_gpt`: train on legacy GPT models, test on newer GPT models.
- `pre5_gpt_to_gpt5`: train on all pre-GPT-5 GPT corpora, test on GPT-5.
- `new_pre5_gpt_to_gpt5`: train only on newer GPT-4-class API data, test on GPT-5.
- `all_pre5_ai_to_gpt5`: train on every available pre-GPT-5 AI source, test on GPT-5.
- `new_non_gpt_to_gpt5`: train on Claude/Gemini, test on GPT-5.

For zero-shot detectors, the AUC is simply evaluated on the transfer test cohort; no detector fitting occurs.


In [ ]:
TRANSFER_ORDER = [
    'old_to_new_ai',
    'old_to_new_gpt',
    'pre5_gpt_to_gpt5',
    'new_pre5_gpt_to_gpt5',
    'all_pre5_ai_to_gpt5',
    'new_non_gpt_to_gpt5',
]

transfer_table = transfer[
    transfer.detector.isin(MAIN_DETECTORS)
].pivot_table(
    index=['human_group','transfer'],
    columns='detector',
    values='auc'
)

transfer_table = transfer_table.reindex(
    pd.MultiIndex.from_product(
        [['old_human','new_human','all_human'], TRANSFER_ORDER],
        names=['human_group','transfer']
    )
)

transfer_table.round(3)


In [ ]:
# Directly inspect the GPT-5 transfer tests.
gpt5_transfer = transfer[
    transfer['transfer'].isin([
        'pre5_gpt_to_gpt5',
        'new_pre5_gpt_to_gpt5',
        'all_pre5_ai_to_gpt5',
        'new_non_gpt_to_gpt5',
    ])
    & transfer.detector.isin(MAIN_DETECTORS)
]

gpt5_transfer.pivot_table(
    index=['human_group','transfer'],
    columns='detector',
    values='auc'
).round(3)


## 6. Trained-NERO variability across repeats

The main tables use the mean trained-NERO AUC over `REPEATS` splits. This cell exposes the individual repeat values so that an apparently strong result is not mistaken for a single favorable split.


In [ ]:
within_repeats.groupby(
    ['human_group','ai_group']
).auc.agg(['mean','std','min','max','count']).sort_values('mean', ascending=False)


In [ ]:
transfer_repeats.groupby(
    ['human_group','transfer']
).auc.agg(['mean','std','min','max','count']).sort_values('mean', ascending=False)


## Interpretation cautions

1. **Binoculars is not available for the legacy tables.** A blank/NaN Binoculars result is missing data, not detector failure.
2. **External detector score availability differs by corpus.** Always check `n_total` before comparing AUCs.
3. **Newer public-domain human tables have many zero HowkGPT/ZeroGPT values.** Those zeros may represent failed or unavailable detector outputs rather than true scores; comparisons using `new_human` therefore require special caution.
4. **NERO_RF is in-distribution in the within-scenario table**, but genuinely cross-generation in the transfer table.
5. **NERO_zero requires no fitting.** It is evaluated directly from the NERO rate vector.
6. For a final manuscript figure, freeze the repository commit, increase `REPEATS` if desired, and report the detector-specific sample counts alongside AUC.
